# ConvNeXt-Small · 224 pixels · fold 0 · Colab
Train **one Small model** using your existing three full 224-pixel cache shards on Drive.
Choose a GPU runtime (A100 if available), set the paths below, and run cells in order.
About **35 GB free local disk** is needed. Original image arrays total 29.35 GB.

Frozen recipe: `convnext_small.fb_in22k_ft_in1k`, 16 slices × five series,
protocol series selection, original sampling, seed 0, 12 epochs, effective batch eight.
**Fold 0 means the current nested outer0 split:** 2,208 training / 553 inner-selection /
691 outer-evaluation studies. It is not the older 4,349-study five-fold split.
Labels and splits are included; no repository upload or label regeneration is needed.
Gold is excluded and never monitored. No 384-pixel cache is used.

A short smoke tests two optimizer steps and checkpoint reload on training images.
Full training starts fresh afterward. Checkpoints and logs save directly to Drive.
Only inner validation selects the checkpoint; outer predictions are produced afterward.

The saved Kaggle Tiny control is a **historical exploratory comparison**: Colab hardware
and runtime can differ. A favorable comparison needs a separate matched confirmation.
No additional folds, blends or submissions run automatically.


In [ ]:
DRIVE_CACHE = "/content/drive/MyDrive/kagglehub"  # Folder containing all three original shards
SIDE_FILE = ""  # Optional full Drive path to side_predictions.parquet if outside DRIVE_CACHE
DRIVE_OUT = "/content/drive/MyDrive/knee_runs/convnext_small_224_outer0_v1"
LOCAL_INPUTS = "/content/knee_small_inputs"


## 1. Setup and mount Drive
Internet is needed for packages and ImageNet pretrained weights. A Kaggle token is needed only if side predictions are missing from Drive.


In [ ]:
%pip install -q "timm==1.0.26" "lightning==2.6.5" kagglehub pyarrow scikit-learn opencv-python-headless "pydicom>=3.0" numpy pandas tqdm
import importlib, shutil, torch
for package in ("timm", "lightning.pytorch", "pyarrow", "sklearn", "cv2", "pydicom", "pandas", "kagglehub"):
    importlib.import_module(package)
if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
    raise RuntimeError("Choose a Colab runtime with exactly one GPU")
from google.colab import drive
drive.mount("/content/drive")
print("GPU:", torch.cuda.get_device_name(0), "| torch:", torch.__version__)
print("Free local disk GiB:", round(shutil.disk_usage('/content').free / 2**30, 1))


## 2. Restore the frozen code and fold
This compressed bundle contains metadata, fixed labels/splits, source and Tiny reference predictions; no MRI images, report text or trained weights.


In [ ]:
import base64, hashlib, io, json, pathlib, sys, zipfile, importlib
BUNDLE_SHA256 = 'f48b5686a230fb747ea32cf4e54b77fc9d37e8442f7cff398ea3c0efa24ff463'
BUNDLE_BASE64 = '...'
CODE_ROOT = pathlib.Path('/content/knee_small_code')
blob = base64.b64decode(BUNDLE_BASE64)
assert hashlib.sha256(blob).hexdigest() == BUNDLE_SHA256
CODE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as bundle:
    for name in bundle.namelist():
        target = (CODE_ROOT / name).resolve()
        if not target.is_relative_to(CODE_ROOT.resolve()):
            raise ValueError('Invalid bundle path')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(bundle.read(name))
manifest = json.loads((CODE_ROOT / 'bundle_manifest.json').read_text())
for name, expected_hash in manifest.items():
    assert hashlib.sha256((CODE_ROOT / name).read_bytes()).hexdigest() == expected_hash, name
sys.path.insert(0, str(CODE_ROOT))
for name in list(sys.modules):
    if name in ('src', 'tools') or name.startswith(('src.', 'tools.')):
        del sys.modules[name]
importlib.invalidate_caches()
from tools.colab_small_runtime import stage_inputs, run_small, run_process
print('Verified bundle:', BUNDLE_SHA256)


## 3. Copy and verify your existing shards
Images are copied to fast Colab local storage. Correct existing local copies are reused. Full SHA-256 checks reject thinned caches, mismatched versions and incomplete copies.


In [ ]:
# Restore missing settings without overwriting your custom paths.
for name, default in {
    'DRIVE_CACHE': '/content/drive/MyDrive/kagglehub',
    'SIDE_FILE': '',
    'DRIVE_OUT': '/content/drive/MyDrive/knee_runs/convnext_small_224_outer0_v1',
    'LOCAL_INPUTS': '/content/knee_small_inputs',
}.items():
    globals().setdefault(name, default)
EXPECTED_INPUTS = {'series_meta': {'path': '/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv', 'sha256': '573c1d80772bf41211c91b149c95677385a1c22d63f485c347f1b46c0177aef3', 'bytes': 3458834}, 'side_predictions': {'path': '/kaggle/input/notebooks/pssv27/knee-side-classifier/side_predictions.parquet', 'sha256': 'd211974a58831d3f6aae781ac4d7e6bd292a19e4cdcfb4341577a31e4db9651c', 'bytes': 1077118}, 'shards': [{'array': {'path': '/kaggle/input/notebooks/pssv27/knee-cache-224-01-shard-1/cache_train_series_00.npy', 'sha256': '393f002ee42c1c9226ef847801a6d31d1438dc3ce76d51c47fd50135608bc7c3', 'bytes': 9672327296}, 'manifest': {'path': '/kaggle/input/notebooks/pssv27/knee-cache-224-01-shard-1/manifest_train_series_00.parquet', 'sha256': '212481fa45499c71c037d7a4d06c3043960e972bf07c094ffbd103e46c5d2488', 'bytes': 413345}}, {'array': {'path': '/kaggle/input/notebooks/pssv27/knee-cache-224-01-shard-2/cache_train_series_01.npy', 'sha256': '802198a895fcc38f6d4967a3d69d61d02843fcf83f68ff65cbe5012a88086cc9', 'bytes': 10180509824}, 'manifest': {'path': '/kaggle/input/notebooks/pssv27/knee-cache-224-01-shard-2/manifest_train_series_01.parquet', 'sha256': 'c0ab434e85496aa56cfbf022b2dc5bc0fd08996cbbd937db72e67541803aa0be', 'bytes': 434330}}, {'array': {'path': '/kaggle/input/notebooks/pssv27/knee-cache-224-01-shard-3/cache_train_series_02.npy', 'sha256': '9ae196c70cb740542404ecd39ae4576a3578b2d3110c58db657799a16a2a994f', 'bytes': 9495306368}, 'manifest': {'path': '/kaggle/input/notebooks/pssv27/knee-cache-224-01-shard-3/manifest_train_series_02.parquet', 'sha256': '5bc7a072cd42b2fc973c9b767f582deae48dea30b71e39c5fd4257b60c6035e5', 'bytes': 406131}}]}
if 'stage_inputs' not in globals() or 'CODE_ROOT' not in globals():
    raise RuntimeError('Run section 2 (Restore the frozen code and fold) before this cell.')
INPUTS = stage_inputs(DRIVE_CACHE, LOCAL_INPUTS, CODE_ROOT, EXPECTED_INPUTS, SIDE_FILE)
# Actual data/role/provenance preflight before any optimizer step.
run_process([sys.executable, '-u', str(CODE_ROOT / 'tools/train_small.py'),
             '--design', str(CODE_ROOT / 'design/candidate'), '--inputs', str(INPUTS)],
            CODE_ROOT, pathlib.Path(LOCAL_INPUTS) / 'preflight.log')


## 4. Smoke test, then train Small once
Selected and last training checkpoints, metrics, predictions and source receipts save to Drive.
The smoke uses transient weights; full training resets the seed and starts from original pretraining.
If the smoke fails or the GPU runs out of memory, execution stops without changing the recipe.

Completed runs are hash-verified and skipped on rerun. A partial run **stops for recovery**,
preserving `last.ckpt` and the selected checkpoint; it does not silently restart fitting.
Do not run two sessions against the same output folder.


In [ ]:
# Restore missing settings without overwriting your custom paths.
for name, default in {
    'DRIVE_CACHE': '/content/drive/MyDrive/kagglehub',
    'SIDE_FILE': '',
    'DRIVE_OUT': '/content/drive/MyDrive/knee_runs/convnext_small_224_outer0_v1',
    'LOCAL_INPUTS': '/content/knee_small_inputs',
}.items():
    globals().setdefault(name, default)
if 'INPUTS' not in globals() or 'run_small' not in globals():
    raise RuntimeError('Run sections 2 and 3 (restore code, then verify shards) before training.')
RUN_DIR = run_small(CODE_ROOT, INPUTS, DRIVE_OUT, BUNDLE_SHA256)


## 5. Results
The exploratory screen requires macro delta ≥0.005, no finding delta below −0.020,
and a positive lower paired-bootstrap bound. A pass is a reason to propose a matched
confirmation, not model promotion or evidence of independent image accuracy.
For review, retain `complete.json`, `comparison.json`, `screen_decision.json`,
`candidate/run_contract.json`, `candidate/outer_predictions.parquet`, training metrics,
and `candidate/selected.pt`. All are saved under DRIVE_OUT.


In [ ]:
print('Saved to:', RUN_DIR)
print((RUN_DIR / 'comparison.json').read_text())
print((RUN_DIR / 'screen_decision.json').read_text())
